# KnowledgeVault -- Week 4 API Notebook

**KnowledgeVault** is a multimodal document-intelligence API built with FastAPI.
It ingests engineering PDFs, describes figures with a vision LLM, and retrieves
text + figure chunks via hybrid search (dense text + BM25, fused with RRF).

---

## Before You Start

1. **Copy `.env.example` to `.env`** and fill in your `OPENAI_API_KEY`.
2. **Start Qdrant** (vector store):  
   `docker run -p 6333:6333 qdrant/qdrant`
3. **Install dependencies**:  
   `pip install -r requirements.txt`
4. **Start the API server** (from the `knowledgevault` folder, the one containing app/ and index.html):  
   `uvicorn app.main:app --reload`
5. **Ingest a PDF** before calling `/retrieve`:  
   `python -m app.ingest --pdf data/sample/attention-is-all-you-need.pdf`


## Cell 1 -- Setup

Run this cell first. It imports everything, sets the base URL, and defines the three
demo queries used throughout the notebook.

In [ ]:
import json
import requests
from pprint import pprint

BASE = 'http://localhost:8000'

# Demo query: text-anchored (Section 2)
DEMO_NOTES = 'Why do the authors use self-attention instead of recurrence in the Transformer?'

# Demo query: figure-anchored (Section 3)
DEMO_FIGURE = 'What does the scaled dot-product attention diagram show?'

# Demo query: out-of-domain (Section 4)
DEMO_OOD = 'What is the best recipe for chocolate chip cookies?'

print('Setup complete.')
print(f'  BASE       = {BASE}')
print(f'  DEMO_NOTES = {DEMO_NOTES}')
print(f'  DEMO_FIGURE= {DEMO_FIGURE}')
print(f'  DEMO_OOD   = {DEMO_OOD}')

---
## Section 1 -- Health Check

`GET /health` is the liveness probe. It returns 200 with model names and the
Qdrant collection name if the server started correctly and `.env` was loaded.
No external API calls are made by this endpoint.

In [ ]:
!curl -s http://localhost:8000/health

In [ ]:
r = requests.get(f'{BASE}/health')
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    pprint(data)

---
## Section 2 -- POST /retrieve -- Text-Anchored Query

### How hybrid retrieval works

KnowledgeVault uses two parallel search channels fused at query time:

| Channel | Method | Best for |
|---------|--------|----------|
| Dense text | `text-embedding-3-large` + Qdrant ANN | Semantic similarity |
| Sparse BM25 | Token frequencies (in-memory) | Exact keyword match |

The two ranked lists are merged using **Reciprocal Rank Fusion (RRF)** with `k=60`.
RRF rewards chunks that appear near the top of both lists without requiring score
normalisation -- dense cosine and BM25 scores live on incompatible scales.

### Request shape

```json
{"query": "<string>", "k": 5, "document_id": null}
```

### Response shape

```json
{
  "query": "<string>",
  "fusion": "rrf",
  "chunks": [
    {
      "chunk_id": "<sha256-prefix>",
      "chunk_type": "prose | figure-description | table-row",
      "text": "<chunk text or figure description>",
      "page_number": 4,
      "section_heading": "<string or null>",
      "image_url": "<url or null>",
      "score": 0.032
    }
  ]
}
```

> `image_url` is set only on `figure-description` chunks and points to
> the extracted PNG served at `/figures/<filename>`.


In [ ]:
!curl -s -X POST "http://localhost:8000/retrieve" -H "Content-Type: application/json" -d "{\"query\": \"Why do the authors use self-attention instead of recurrence in the Transformer?\", \"k\": 5}"

In [ ]:
payload = {'query': DEMO_NOTES, 'k': 5, 'document_id': None}
r = requests.post(f'{BASE}/retrieve', json=payload)
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'Query  : {data["query"]}')
    print(f'Fusion : {data["fusion"]}')
    print(f'Chunks : {len(data["chunks"])}')
    print()
    for i, chunk in enumerate(data['chunks']):
        print(f'[{i+1}] score={chunk["score"]:.4f}  type={chunk["chunk_type"]}  page={chunk["page_number"]}')
        print(f'     {chunk["text"][:120]}...' if len(chunk['text']) > 120 else f'     {chunk["text"]}')
        print()

---
## Section 3 -- POST /retrieve -- Figure-Anchored Query

Figure chunks are indexed using the same `text-embedding-3-large` embedder as prose.
The text that gets embedded is the vision LLM's structured description:

```
[architecture diagram] This figure shows the Transformer model architecture
with an encoder stack on the left and a decoder stack on the right ...
Key elements: Multi-Head Attention, Add & Norm, Feed Forward, Positional Encoding
```

When you ask about a diagram, your query semantically matches that description
and the figure-description chunk surfaces in the fused results.

Look for `chunk_type == 'figure-description'` in the output. Each one carries
an `image_url` field pointing to the extracted figure PNG at `/figures/<filename>`.


In [ ]:
!curl -s -X POST "http://localhost:8000/retrieve" -H "Content-Type: application/json" -d "{\"query\": \"What does the scaled dot-product attention diagram show?\", \"k\": 5}"

In [ ]:
payload = {'query': DEMO_FIGURE, 'k': 5, 'document_id': None}
r = requests.post(f'{BASE}/retrieve', json=payload)
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'Query  : {data["query"]}')
    print(f'Fusion : {data["fusion"]}')
    print(f'Chunks : {len(data["chunks"])}')
    print()
    for i, chunk in enumerate(data['chunks']):
        image_url = chunk.get('image_url')
        print(f'[{i+1}] score={chunk["score"]:.4f}  type={chunk["chunk_type"]}  page={chunk["page_number"]}')
        if image_url:
            print(f'     image_url = {image_url}')
        print(f'     {chunk["text"][:120]}...' if len(chunk['text']) > 120 else f'     {chunk["text"]}')
        print()

---
## Section 4 -- POST /retrieve -- Out-of-Domain Query

When the query has no semantic overlap with any indexed document, the retriever
**still returns the top-k closest matches it can find** -- there is no score
threshold in `retriever.py`, so you get back k chunks with uniformly low RRF
scores rather than an empty list.

That is the honest behavior of this layer: retrieval ranks, it does not refuse.
The low scores across the board are the signal a downstream consumer should use
to say "I don't know" instead of inventing an answer.

> **Forward note:** Week 5's CitationRAG adds exactly that -- a score-threshold
> gate on top of `/retrieve` plus a refusal path when nothing clears it.


In [ ]:
!curl -s -X POST "http://localhost:8000/retrieve" -H "Content-Type: application/json" -d "{\"query\": \"What is the best recipe for chocolate chip cookies?\", \"k\": 5}"

In [ ]:
payload = {'query': DEMO_OOD, 'k': 5, 'document_id': None}
r = requests.post(f'{BASE}/retrieve', json=payload)
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'Query  : {data["query"]}')
    print(f'Chunks : {len(data["chunks"])}')
    print()
    for i, chunk in enumerate(data['chunks']):
        print(f'[{i+1}] score={chunk["score"]:.4f}  type={chunk["chunk_type"]}  page={chunk["page_number"]}')
    print()
    print('Note the uniformly low RRF scores -- the retriever returns its top-k')
    print('closest-but-still-bad matches; it has no threshold and never refuses.')
    print("Week 5's CitationRAG adds the score-threshold gate + refusal on top.")


---
## Section 5 -- POST /retrieve -- With document_id Filter

The `document_id` field lets you **pre-filter** the vector search to a single document
before scoring. This is more efficient than post-filtering because Qdrant applies the
filter at the ANN search level, not after fetching all candidates.

**Pre-filter vs post-filter:**
- **Pre-filter** (used here): The filter is pushed into the ANN query. Only vectors
  belonging to `attention-is-all-you-need` are considered as candidates. Faster and more accurate
  at high k.
- **Post-filter**: Fetch `k * n` candidates, then discard those not matching. Wastes
  recall budget and is slower.

In [ ]:
payload = {
    'query': DEMO_NOTES,
    'k': 5,
    'document_id': 'attention-is-all-you-need'
}
r = requests.post(f'{BASE}/retrieve', json=payload)
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'Query       : {data["query"]}')
    print(f'document_id : attention-is-all-you-need (pre-filter applied)')
    print(f'Chunks      : {len(data["chunks"])}')
    print()
    for i, chunk in enumerate(data['chunks']):
        print(f'[{i+1}] score={chunk["score"]:.4f}  type={chunk["chunk_type"]}  page={chunk["page_number"]}')
        print(f'     {chunk["text"][:120]}...' if len(chunk['text']) > 120 else f'     {chunk["text"]}')
        print()

---
## Section 6 -- POST /ingest -- Ingest a Single PDF

Ingestion runs a five-stage pipeline:

1. **Parse** -- PyMuPDF extracts text blocks, figure crops (saved as PNGs), and table regions
2. **Describe** -- each figure crop is sent to `gpt-5.4-mini-2026-03-17` via forced function-calling;
   low-confidence descriptions are dropped before chunking
3. **Chunk** -- prose into 300-token sliding windows; figures into one chunk per figure;
   tables into one chunk per row with column-header prefixes
4. **Embed** -- `text-embedding-3-large` (3072-dim) for every chunk
5. **Index** -- chunks + vectors upserted into the `knowledgevault` Qdrant collection

### Preferred method: CLI

Run from the `knowledgevault` folder, the one containing app/ and index.html -- gives live stage-by-stage logging:

```
python -m app.ingest --pdf data/sample/attention-is-all-you-need.pdf
python -m app.ingest --pdf data/sample/attention-is-all-you-need.pdf --reset
```

`--reset` drops and recreates the Qdrant collection before ingesting (clean slate).

### HTTP endpoint

The endpoint accepts `pdf_path` as a **query parameter** (not a JSON body).
For large PDFs, prefer the CLI to avoid HTTP timeouts.


In [ ]:
!curl -s -X POST "http://localhost:8000/ingest?pdf_path=data/sample/attention-is-all-you-need.pdf"

In [ ]:
r = requests.post(f'{BASE}/ingest',
                  params={'pdf_path': 'data/sample/attention-is-all-you-need.pdf'})
if r.status_code != 200:
    print(f'Status {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'ok     : {data["ok"]}')
    pprint(data.get('summary', {}))


---
## Section 7 -- GET /pdfs -- List Available PDFs

Lists all PDF files currently in the data directory (`data/sample/`).  
Use this to check what has been uploaded before ingesting.

Response shape:
```json
{ "directory": "data/sample", "count": 2, "pdfs": [{"filename": "...", "size_kb": 120.5}] }
```


In [ ]:
!curl -s http://localhost:8000/pdfs

In [ ]:
r = requests.get(f'{BASE}/pdfs')
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print(f'Directory : {data["directory"]}')
    print(f'PDF count : {data["count"]}')
    print()
    for pdf in data['pdfs']:
        print(f'  {pdf["filename"]}  ({pdf["size_kb"]} KB)')

---
## Section 8 -- POST /upload -- Upload a PDF

Upload a local PDF file to the server's data directory so it can be ingested.  
Uses multipart form-data. After upload, the file appears in `GET /pdfs`.

> Requires a real PDF file on your machine. Update `PDF_PATH` below.

Response shape:
```json
{ "filename": "runbook.pdf", "path": "data/sample/runbook.pdf", "size_bytes": 102400 }
```


In [ ]:
# Update this path to a PDF on your machine
PDF_PATH = 'data/sample/attention-is-all-you-need.pdf'   # change as needed

import os
if not os.path.exists(PDF_PATH):
    print(f'File not found: {PDF_PATH}')
    print('Update PDF_PATH above to point to a real PDF file.')
else:
    with open(PDF_PATH, 'rb') as f:
        r = requests.post(f'{BASE}/upload', files={'file': (os.path.basename(PDF_PATH), f, 'application/pdf')})
    if r.status_code != 200:
        print(f'Error {r.status_code}:', r.json())
    else:
        data = r.json()
        print('Uploaded   :', data['filename'])
        print('Saved to   :', data['path'])
        print('Size (KB)  :', round(data['size_bytes'] / 1024, 1))

---
## Section 9 -- POST /ingest/all -- Ingest All PDFs

Ingests every PDF in the data directory in one call.  
Runs the full pipeline (parse → describe → chunk → embed → upsert) for each file.

Key concept: `reset=true` drops and recreates the Qdrant collection before the **first** PDF only,  
so subsequent files append to the same collection rather than overwriting each other.

Response shape:
```json
{
  "ok": true,
  "summaries": [{"pdf": "...", "blocks": 42, "chunks": 87, "figures_described": 4, "figures_dropped": 1}],
  "errors": []
}
```

> Heavy operation - each figure makes a real vision API call. Start with a small PDF.


In [ ]:
!curl -s -X POST http://localhost:8000/ingest/all

In [ ]:
# Add reset=true to drop and recreate the Qdrant collection first
r = requests.post(f'{BASE}/ingest/all', params={'reset': 'false'})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print(f'ok      : {data["ok"]}')
    print(f'PDFs    : {len(data["summaries"])}')
    print(f'Errors  : {len(data["errors"])}')
    print()
    for s in data['summaries']:
        name = s['pdf'].split('/')[-1]
        print(f'  {name}')
        print(f'    blocks           : {s["blocks"]}')
        print(f'    chunks indexed   : {s["chunks"]}')
        print(f'    figures described: {s["figures_described"]}')
        print(f'    figures dropped  : {s["figures_dropped"]}')
    for err in data['errors']:
        print(f'  ERROR {err["pdf"]}: {err["error"]}')

---
## Section 10 -- Full Raw Response Dump

Print the complete JSON from `POST /retrieve` with `DEMO_NOTES`, untruncated.
Useful for inspecting every field in `chunks` before building a UI on top.


In [ ]:
payload = {'query': DEMO_NOTES, 'k': 5, 'document_id': None}
r = requests.post(f'{BASE}/retrieve', json=payload)
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(json.dumps(data, indent=2))

---
## Section 11 -- Failure Mode: Qdrant Not Running or Empty Collection (502)

If Qdrant is not running, or if you call `/retrieve` before ingesting any documents,
the retriever raises a connection or collection error which FastAPI converts to a
**502 Bad Gateway**.

**How to trigger:**
- Call `/retrieve` without Qdrant running
- Or call `/retrieve` before any `/ingest` call (collection does not exist yet)

**Fix:**  
1. Make sure Qdrant is running: `docker run -p 6333:6333 qdrant/qdrant`
2. Ingest at least one PDF: `python -m app.ingest --pdf data/sample/attention-is-all-you-need.pdf`

> The server stays up and returns a readable error -- it does not crash.


In [ ]:
!curl -s -X POST "http://localhost:8000/retrieve" -H "Content-Type: application/json" -d "{\"query\": \"Why do the authors use self-attention instead of recurrence?\", \"k\": 5}"

In [ ]:
# Trigger a 502 by querying before ingesting (collection does not exist yet).
# Make sure you have NOT run /ingest before this cell.
r = requests.post(f'{BASE}/retrieve', json={'query': DEMO_NOTES, 'k': 5})
print(f'Status code : {r.status_code}')
data = r.json()
print(f'Detail      : {data.get("detail", "no detail")}')
if r.status_code == 502:
    print()
    print('Expected 502 -- Qdrant not running or no documents ingested yet.')
    print('Fix: start Qdrant, then run /ingest before calling /retrieve.')


---
## Section 12 -- Failure Mode: Invalid Request (422)

Pydantic validates every incoming request body. If the required `query` field is
missing or null, FastAPI returns **422 Unprocessable Entity** with a structured
validation error -- not a 500, not a silent failure.

This demonstrates the value of a schema-first API: bad inputs are caught at the
boundary before they reach any business logic.

In [ ]:
!curl -s -X POST "http://localhost:8000/retrieve" -H "Content-Type: application/json" -d "{\"k\": 5}"

In [ ]:
# Send a payload with no 'query' field to trigger Pydantic validation error
payload = {'k': 5}
r = requests.post(f'{BASE}/retrieve', json=payload)
print(f'Status code : {r.status_code}')
data = r.json()
print(f'Response    :')
print(json.dumps(data, indent=2))
if r.status_code == 422:
    print()
    print('Expected 422 -- Pydantic caught the missing required field "query".')

---
## Section 13 -- OpenAPI / Swagger Docs

FastAPI auto-generates interactive API documentation from the route definitions
and Pydantic schemas. Open these in your browser while the server is running:

- **Swagger UI** (try endpoints interactively):  
  [http://localhost:8000/docs](http://localhost:8000/docs)

- **ReDoc** (clean reference view):  
  [http://localhost:8000/redoc](http://localhost:8000/redoc)

- **Raw OpenAPI JSON schema**:  
  [http://localhost:8000/openapi.json](http://localhost:8000/openapi.json)

In [ ]:
r = requests.get(f'{BASE}/openapi.json')
if r.status_code != 200:
    print(f'ERROR {r.status_code}: {r.text}')
else:
    data = r.json()
    print(f'API title   : {data["info"]["title"]}')
    print(f'API version : {data["info"]["version"]}')
    print(f'Routes      : {list(data["paths"].keys())}')
    print()
    print(f'Swagger UI  --> http://localhost:8000/docs')
    print(f'ReDoc       --> http://localhost:8000/redoc')